In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
splcher_animefacedataset_path = kagglehub.dataset_download('splcher/animefacedataset')

print('Data source import complete.')


> ⚠️ **AVISO TÉCNICO: HIGIENIZAÇÃO DE OUTPUTS (I/O LIMITS)**
>
> *Atenção Avaliador:* Este caderno original atingiu a marca de **51 MB** de tamanho. O formato `.ipynb` codifica imagens de saída (outputs) em *Base64* dentro do JSON. Ao gerar e salvar imagens de alta resolução ao longo de 150 épocas, o limite de buffer do Kaggle/Colab foi excedido, gerando o erro de parse (`Unterminated string in JSON`).
>
> Para garantir a integridade e a revisão do código-fonte, **as saídas (outputs) visuais deste caderno foram intencionalmente limpas**. Abaixo, apresentamos os logs de treinamento que comprovam a convergência matemática da rede ao longo do tempo.
>
> ---
>
> ### 📊 Resumo do Treinamento (Gerador Peso-Pesado vs. Policial Implacável)
>
> **Início do Treinamento (Época 1): O Choque Inicial**
> O Gerador, ainda aleatório, tenta enganar o Discriminador sem sucesso, resultando em um domínio avassalador do Policial.
> * `[1/150][0/497] Loss_D: 1.388 | Loss_G: 0.002 || D(x): 0.610 | D(G(z)): 0.998 -> Falsificador Dominando` (Ruído estocástico inicial)
> * `[1/150][400/497] Loss_D: 1.219 | Loss_G: 1.909 || D(x): 0.495 | D(G(z)): 0.177 -> Equilíbrio Tenso`
>
> **Fim do Treinamento (Época 150): O Estado da Arte Local**
> O Gerador (com 128 feature maps) resiste à pressão de um Discriminador não-suavizado, mantendo um gradiente saudável e forçando um equilíbrio matemático tenso.
> * `[150/150][0/497] Loss_D: 0.585 | Loss_G: 2.474 || D(x): 0.766 | D(G(z)): 0.102 -> Equilíbrio Tenso`
> * `[150/150][400/497] Loss_D: 0.458 | Loss_G: 2.802 || D(x): 0.842 | D(G(z)): 0.076 -> Equilíbrio Tenso`
>
> *(Nota: As imagens geradas na Época 1 e na Época 150 estão documentadas e indexadas no repositório deste projeto para avaliação visual).*

# ==============================================================================
# PROJETO 4: DEEP CONVOLUTIONAL GENERATIVE ADVERSARIAL NETWORKS (DCGANs)
# A Batalha Minimax: Criação de Dados In-The-Wild
# ==============================================================================

## 1. Introdução à Teoria Adversarial

Diferente de modelos tradicionais que buscam minimizar uma única função de perda, as Redes Adversárias Generativas (GANs), introduzidas por Ian Goodfellow em 2014, operam sob a lógica da **Teoria dos Jogos**. Consiste no treinamento simultâneo de duas redes neurais distintas que competem em um jogo de soma zero (*Minimax Game*):

* **O Gerador ($G$):** Atua como um "falsificador". Sua missão é receber um vetor de ruído aleatório (Espaço Latente $z$) e transformá-lo em uma imagem sintética tão realista que seja impossível distingui-la da realidade.
* **O Discriminador ($D$):** Atua como o "policial" ou perito. É um classificador binário cuja missão é analisar uma imagem e determinar a probabilidade dela ser **Real** (vinda do dataset) ou **Falsa** (criada pelo Gerador).

A equação fundamental que rege este cabo de guerra matemático é:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1 - D(G(z)))]$$

## 2. O Desafio do Equilíbrio e o Colapso

A arquitetura original do professor foi baseada em Redes Densas para imagens de baixa resolução ($28 \times 28$, em tons de cinza). Para este projeto, elevaremos a complexidade e implementaremos uma **DCGAN (Deep Convolutional GAN)** para lidar com imagens coloridas (RGB) em resolução $64 \times 64$.

A dificuldade central deste projeto não é apenas escrever a arquitetura, mas manter a **Homeostase do Treinamento**. Se uma rede superar a outra muito rapidamente, enfrentaremos falhas catastróficas:
1.  **Fuga de Gradiente (Vanishing Gradients):** Se o Discriminador ficar muito "inteligente" rápido demais, ele rejeitará as imagens do Gerador com 100% de certeza. O gradiente de aprendizado cai a zero, e o Gerador para de aprender.
2.  **Colapso de Modo (Mode Collapse):** Se o Gerador for muito forte e achar uma "falha" no Discriminador, ele passará a gerar *exatamente a mesma imagem* repetidas vezes, pois descobriu que aquela imagem específica sempre engana o policial.

## 3. O Espaço Latente ($z$)

A entrada do nosso Gerador será o **Vetor Latente** (ex: `latent_dim = 100`).
* Não devemos pensar nele apenas como "100 números aleatórios", mas como as **coordenadas em um espaço multidimensional de características**.
* Durante o treinamento, a DCGAN forçará esse espaço a ganhar significado semântico. Por exemplo, uma direção no espaço latente pode representar a "cor do cabelo", enquanto outra direção pode representar "iluminação". Ao navegar suavemente por esse espaço, a IA será capaz de interpolar imagens, criando transições contínuas entre feições distintas.

# ==============================================================================
# PROJETO 4: DEEP CONVOLUTIONAL GENERATIVE ADVERSARIAL NETWORKS (DCGANs)
# ==============================================================================

### **Célula 1: Setup Automatizado, Hiperparâmetros e Ingestão de Dados**

Nesta primeira etapa, construímos um pipeline de dados agnóstico e totalmente automatizado utilizando a API `kagglehub`. Em vez de dependermos de caminhos absolutos fixos que mudam entre computadores locais e instâncias de nuvem, o pipeline baixa o dataset `splcher/animefacedataset` (composto por mais de 63.000 rostos alinhados humanos), mapeia o diretório dinamicamente e prepara os lotes de tensores na GPU.

#### **Análise dos Hiperparâmetros da DCGAN Clássica (Artigo de Radford et al.):**

* `BATCH_SIZE = 128`: O tamanho do lote define quantas imagens o Discriminador e o Gerador analisarão antes de atualizar seus pesos. O valor de 128 fornece estabilidade para o cálculo do gradiente em jogos Minimax.
* `IMAGE_SIZE = 64`: Redimensiona as imagens coloridas para $64 \times 64$ pixels. É a resolução padrão-ouro para DCGANs tradicionais, oferecendo um equilíbrio perfeito entre complexidade estrutural e viabilidade computacional na GPU Tesla T4.
* `LATENT_DIM = 100`: A dimensão do espaço latente. O Gerador receberá um vetor de 100 números aleatórios (ruído gaussiano) e usará essa semente multidimensional para aprender a mapear características como expressões, cores de cabelo e orientações faciais.
* `LR = 0.0002` e `BETA1 = 0.5`: Parâmetros críticos do otimizador Adam. O artigo original da DCGAN comprovou que reduzir o `beta1` de 0.9 (padrão) para 0.5 remove a oscilação violenta do gradiente, permitindo que o Gerador e o Discriminador aprendam em um ritmo competitivo emparelhado.

#### **Pipeline de Transformação Visual:**
Os dados sofrem uma normalização estrita para o intervalo de $[-1.0, 1.0]$ através de `transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))`. Essa etapa é estritamente obrigatória porque a última camada do nosso Gerador utilizará a função de ativação **Tangente Hiperbólica (Tanh)**, cujo output matemático varia nativamente entre $-1$ e $1$.

In [ ]:
# ==============================================================================
# BLOCO 1: DOWNLOAD AUTOMÁTICO, HIPERPARÂMETROS E PIPELINE DE DADOS (PYTORCH)
# ==============================================================================

import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

# Garante a semente aleatória estrita para reprodutibilidade dos experimentos
SÉMENTE_CIENTIFICA = 42
random.seed(SÉMENTE_CIENTIFICA)
torch.manual_seed(SÉMENTE_CIENTIFICA)
np.random.seed(SÉMENTE_CIENTIFICA)

# ------------------------------------------------------------------------------
# 1. DOWNLOAD E CONFIGURAÇÃO DINÂMICA DE DIRETÓRIO via KAGGLEHUB
# ------------------------------------------------------------------------------
print("Iniciando verificação/download programático do Anime Face Dataset...")
# Baixa e extrai automaticamente a versão mais recente do dataset para o cache
dataset_download_path = kagglehub.dataset_download("splcher/animefacedataset")
print(f"✅ Dataset localizado em: {dataset_download_path}")

# Definição do DATAROOT dinâmico baseado no retorno do kagglehub
DATAROOT = dataset_download_path

# ------------------------------------------------------------------------------
# 2. DEFINIÇÃO DOS HIPERPARÂMETROS ESTRUTURAIS
# ------------------------------------------------------------------------------
WORKERS = 2           # Número de threads paralelas da CPU para o I/O de dados
BATCH_SIZE = 128      # Tamanho do lote para estabilização do gradiente Minimax
IMAGE_SIZE = 64       # Resolução espacial estrita para o funil convolucional (64x64)
CHANNELS = 3          # Imagens coloridas (Canais RGB)
LATENT_DIM = 100      # Dimensão do vetor z de entrada (Espaço Latente)
FEATURES_G = 128       # Escala de profundidade dos mapas de características do Gerador
FEATURES_D = 64       # Escala de profundidade dos mapas de características do Discriminador
NUM_EPOCHS = 150       # Quantidade de épocas para o treinamento inicial
LR = 0.0002           # Taxa de aprendizado recomendada pelo artigo DCGAN
BETA1 = 0.5           # Amortecedor do momentum do Adam contra oscilações violentas

# Inicialização do Hardware (Usa aceleração por GPU se disponível no Kaggle)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Hardware de processamento ativo: {device}")

# ------------------------------------------------------------------------------
# 3. PIPELINE DE TENSORES E INSTANCIAÇÃO DO DATALOADER
# ------------------------------------------------------------------------------
transformacoes = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),           # Ajusta a escala proporcionalmente
    transforms.CenterCrop(IMAGE_SIZE),       # Garante o corte quadrado exato de 64x64
    transforms.ToTensor(),                   # Mapeia os pixels de [0, 255] para [0.0, 1.0]
    transforms.Normalize((0.5, 0.5, 0.5),    # Centraliza o espectro RGB no intervalo [-1.0, 1.0]
                         (0.5, 0.5, 0.5))    # Alinhamento mandatório para a ativação Tanh do Gerador
])

# Carrega a estrutura de pastas usando o PyTorch ImageFolder
dataset = dset.ImageFolder(root=DATAROOT, transform=transformacoes)

# Cria o DataLoader gerenciador de lotes assíncronos
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS
)

print(f"Pipeline concluído! Total de imagens prontas para treino: {len(dataset)}")

# ------------------------------------------------------------------------------
# 4. INSPEÇÃO E AUDITORIA VISUAL DO DATASET
# ------------------------------------------------------------------------------
# Extrai um único lote de teste do pipeline para validar as dimensões e cores
real_batch = next(iter(dataloader))
print(f"Formato dimensional do lote de tensores: {real_batch[0].shape}")

plt.figure(figsize=(10, 10))
plt.axis("off")
plt.title("Amostra de Ingestão: Imagens Reais Alinhadas (Fashion/Anime Domain)")
# Desfaz a normalização de forma temporária apenas para a plotagem humana do matplotlib
grid_img = vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu()
plt.imshow(np.transpose(grid_img, (1, 2, 0)))
plt.show()

## 5. A Arquitetura do Falsificador (O Gerador)

O papel do Gerador ($G$) em uma DCGAN não é comprimir dados, mas **expandi-los**. Ele recebe um vetor de ruído comprimido (nosso espaço latente de 100 dimensões) e aplica uma série de "Convoluções Transpostas" (também chamadas de Deconvoluções) para aumentar a largura e altura do tensor iterativamente, até atingir a resolução de $64 \times 64$ pixels em 3 canais de cor (RGB).

#### **A Topologia do Gerador (Diretrizes de Radford et al.):**
Para evitar que a rede sofra de colapso de gradiente, aplicamos estritamente as regras de arquitetura do paper original da DCGAN:
1.  **Inicialização de Pesos Customizada:** Em vez da inicialização padrão do PyTorch, forçamos todas as convoluções a iniciarem com uma distribuição Normal de média $0.0$ e desvio padrão $0.02$. Isso é vital para que a "briga" entre Gerador e Discriminador comece equilibrada.
2.  **Sem Camadas Densas (Fully Connected):** O modelo é $100\%$ convolucional.
3.  **Batch Normalization:** Aplicado após cada convolução transposta (exceto na saída). Isso estabiliza o aprendizado, garantindo que o tensor não exploda com valores numéricos absurdos durante a expansão.
4.  **Funções de Ativação:** Usamos a `ReLU` em todas as camadas ocultas do Gerador para promover a esparsidade e o fluxo rápido de gradientes.
5.  **A Camada de Saída (Tanh):** A última camada do gerador usa obrigatoriamente a função Tangente Hiperbólica (`Tanh`), que comprime os pixels finais para a escala matemática de $[-1.0, 1.0]$. Como nosso *DataLoader* ajustou as imagens reais exatamente para essa mesma escala, o "policial" (Discriminador) fará uma comparação estatisticamente justa.

A projeção geométrica do tensor dentro da rede ocorrerá na seguinte ordem espacial:
Vetor Latente $\rightarrow 4 \times 4 \rightarrow 8 \times 8 \rightarrow 16 \times 16 \rightarrow 32 \times 32 \rightarrow 64 \times 64$

In [ ]:
# ==============================================================================
# BLOCO 2: INICIALIZAÇÃO DE PESOS E ARQUITETURA DO GERADOR
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. FUNÇÃO DE INICIALIZAÇÃO DE PESOS (REGRA DA DCGAN)
# ------------------------------------------------------------------------------
def weights_init(m):
    """
    Aplica a inicialização customizada aos pesos das camadas Convolucionais e de Normalização.
    Segundo Radford et al., inicializar com uma Normal (mean=0.0, std=0.02) previne
    o travamento prematuro da rede adversária nas primeiras épocas.
    """
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        # Se for uma camada Convolucional (Conv2d ou ConvTranspose2d)
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        # Se for uma camada de Normalização de Lote (Batch Normalization)
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# ------------------------------------------------------------------------------
# 2. CLASSE DO GERADOR (G)
# ------------------------------------------------------------------------------
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()

        # A rede Sequential processa o vetor de ruído em cascata
        self.main = nn.Sequential(
            # Entrada: Vetor Z latente (LATENT_DIM x 1 x 1)
            # Primeira expansão: Gera um bloco 4x4 denso
            nn.ConvTranspose2d(LATENT_DIM, FEATURES_G * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(FEATURES_G * 8),
            nn.ReLU(True),

            # Estado atual: (FEATURES_G*8) x 4 x 4
            # Segunda expansão: Dobra o tamanho para 8x8
            nn.ConvTranspose2d(FEATURES_G * 8, FEATURES_G * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(FEATURES_G * 4),
            nn.ReLU(True),

            # Estado atual: (FEATURES_G*4) x 8 x 8
            # Terceira expansão: Dobra o tamanho para 16x16
            nn.ConvTranspose2d(FEATURES_G * 4, FEATURES_G * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(FEATURES_G * 2),
            nn.ReLU(True),

            # Estado atual: (FEATURES_G*2) x 16 x 16
            # Quarta expansão: Dobra o tamanho para 32x32
            nn.ConvTranspose2d(FEATURES_G * 2, FEATURES_G, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(FEATURES_G),
            nn.ReLU(True),

            # Estado atual: FEATURES_G x 32 x 32
            # Quinta e última expansão: Dobra para a resolução final 64x64
            # O número de canais de saída cai para 3 (RGB)
            nn.ConvTranspose2d(FEATURES_G, CHANNELS, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh() # Comprime os valores finais da imagem para o espectro [-1.0, 1.0]
        )

    def forward(self, input):
        """Método de propagação frontal."""
        return self.main(input)

# ------------------------------------------------------------------------------
# 3. INSTANCIAÇÃO DO MODELO GERADOR
# ------------------------------------------------------------------------------
# Cria a rede e a envia para a memória da GPU (ou CPU)
netG = Generator().to(device)

# Aplica a regra de inicialização estatística aos pesos da rede recém-criada
netG.apply(weights_init)

# Imprime o sumário da arquitetura do Gerador para auditoria
print("=== ARQUITETURA DO GERADOR (FALSIFICADOR) ===")
print(netG)

## 6. A Arquitetura do Perito (O Discriminador)

O Discriminador (D) é o arqui-inimigo do Gerador. Sua topologia é, na prática, o espelho reverso do Gerador. Ele atua como um classificador binário convolucional: recebe uma imagem **64x64** em 3 canais (RGB) e aplica uma série de convoluções para extrair características, reduzindo a dimensão espacial pela metade a cada camada, até sobrar apenas um único valor probabilístico.

#### **A Engenharia Reversa (Diretrizes de Radford et al.):**

Para manter o Policial em pé de igualdade com o Falsificador, algumas regras arquiteturais estritas foram seguidas:
1. **Redução Aprendida (Sem Max Pooling):** Em redes classificadoras comuns, usamos `MaxPooling` para reduzir a imagem. Aqui, o rebaixamento da resolução é feito exclusivamente por camadas `Conv2d` com passo (**stride**) igual a 2. Isso obriga a rede a aprender sua própria função de redução espacial, retendo características anatômicas vitais do anime.
2. **Prevenção de Gradientes Mortos (LeakyReLU):** Diferente do Gerador que usa a ReLU padrão, o Discriminador utiliza a `LeakyReLU` (com inclinação negativa de **0.2**). Em GANs, a ReLU comum pode "zerar" conexões inteiras prematuramente, criando buracos negros de informação. A LeakyReLU permite um leve vazamento de sinal, garantindo que o Gerador sempre receba um feedback matemático de volta.
3. **Ponto Cego Calculado (Sem BatchNorm Inicial):** Não aplicamos Normalização de Lote na primeira camada. Isso permite que a rede aprenda a distribuição real de luz e contraste do dataset logo no primeiro contato, sem que os tensores sejam artificialmente centralizados.
4. **Veredito Final (Sigmoid):** A última ativação comprime o valor final para o intervalo exato de **0 a 1**, representando a probabilidade estatística absoluta da imagem ser real (1) ou falsa (0).

In [ ]:
# ==============================================================================
# BLOCO 3: ARQUITETURA DO DISCRIMINADOR (POLICIAL) E INICIALIZAÇÃO
# ==============================================================================

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # ------------------------------------------------------------------
            # CAMADA 1: Entrada (3 canais RGB) -> Saída (64 mapas)
            # A imagem reduz de 64x64 para 32x32 pixels
            # ------------------------------------------------------------------
            nn.Conv2d(CHANNELS, FEATURES_D, kernel_size=4, stride=2, padding=1, bias=False),
            # Omissão deliberada do BatchNorm na primeira camada
            nn.LeakyReLU(0.2, inplace=True),

            # ------------------------------------------------------------------
            # CAMADA 2: Entrada (64 mapas) -> Saída (128 mapas)
            # A imagem reduz de 32x32 para 16x16 pixels
            # ------------------------------------------------------------------
            nn.Conv2d(FEATURES_D, FEATURES_D * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(FEATURES_D * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # ------------------------------------------------------------------
            # CAMADA 3: Entrada (128 mapas) -> Saída (256 mapas)
            # A imagem reduz de 16x16 para 8x8 pixels
            # ------------------------------------------------------------------
            nn.Conv2d(FEATURES_D * 2, FEATURES_D * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(FEATURES_D * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # ------------------------------------------------------------------
            # CAMADA 4: Entrada (256 mapas) -> Saída (512 mapas)
            # A imagem reduz de 8x8 para 4x4 pixels
            # ------------------------------------------------------------------
            nn.Conv2d(FEATURES_D * 4, FEATURES_D * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(FEATURES_D * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # ------------------------------------------------------------------
            # CAMADA 5 (SAÍDA): Entrada (512 mapas) -> Saída (1 escalar binário)
            # O kernel 4x4 consome a matriz inteira, resultando em 1x1
            # ------------------------------------------------------------------
            nn.Conv2d(FEATURES_D * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid() # Converte a saída para uma probabilidade [0, 1]
        )

    def forward(self, input):
        """Fluxo de propagação do Perito."""
        return self.main(input)

# Instanciação e alocação na GPU
netD = Discriminator().to(device)

# Aplica a regra estatística (Normal mean=0, std=0.02) nos pesos iniciais
netD.apply(weights_init)

print("=== ARQUITETURA DO DISCRIMINADOR (PERITO/POLICIAL) ===")
print(netD)

## 7. Preparação da Arena (Loss, Otimizadores e Ruído Fixo)

Antes de iniciarmos a batalha, definimos as "regras do jogo".

#### **O que estamos modificando em relação ao código base:**
1. **A Função de Perda:** Mantemos a **Binary Cross Entropy (BCELoss)**, que é o padrão da indústria para medir erros de probabilidade binária (0 ou 1). Ela punirá o Discriminador se ele classificar um anime real como falso, e punirá o Gerador se o Discriminador pegar a falsificação.
2. **Os Otimizadores Adam (Aceleradores Estocásticos):** O código do professor já trazia a intuição correta, mas nós a reforçamos. Usamos o otimizador Adam com `lr=0.0002` e `beta1=0.5`. O `beta1` reduzido atua como um "freio de inércia". Em GANs, se o otimizador pegar muito impulso na direção errada, a rede colapsa. O valor de $0.5$ garante que a rede mude de ideia rapidamente caso o oponente mude de tática.
3. **A Geometria do Ruído Fixo (`fixed_noise`):** No código do professor, o ruído era passado como um vetor 2D `(batch, 100)`. Como nossa nova arquitetura é $100\%$ Convolucional (usando `ConvTranspose2d`), os tensores exigem geometria espacial Rank-4. Por isso, geramos nosso ruído fixo no formato `(64, 100, 1, 1)` — ou seja, 64 amostras, de 100 canais, com $1 \times 1$ pixel de tamanho. Esse ruído será congelado e usado como "semente" visual ao longo de todas as 100 épocas.

In [ ]:
# ==============================================================================
# BLOCO 4: FUNÇÃO DE PERDA, OTIMIZADORES E SEMENTE VISUAL
# ==============================================================================

criterion = nn.BCELoss()

# Semente visual congelada (Ruído Fixo)
fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)

# ------------------------------------------------------------------------------
# ATUALIZAÇÃO TÉCNICA: LABEL SMOOTHING (Suavização de Rótulos)
# ------------------------------------------------------------------------------
# Em vez de 1.0 (certeza absoluta), usamos 0.9. Isso "nerfa" o Discriminador,
# impedindo que ele esmague os gradientes do Gerador ao longo de 350 épocas.
real_label = 0.9
fake_label = 0.0

optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))

os.makedirs("evolucao_dcgan", exist_ok=True)
os.makedirs("modelos_salvos", exist_ok=True) # Pasta para salvar os Pesos da IA

print("Regras do Minimax estabelecidas. Label Smoothing Ativado!")

## 8. A Batalha Minimax (Treinamento Adversarial)

Este é o laço de execução principal. Para cada lote de imagens no dataset, executamos estritamente duas fases em sequência:

**Fase 1: O Turno do Discriminador (Atualizar a rede D)**
* Primeiro, o Policial recebe um lote de imagens **Reais** do anime e as avalia (esperando o valor probabilístico de 1.0).
* Em seguida, o Falsificador (Gerador) cria imagens **Falsas** a partir do ruído latente. O Policial as avalia (esperando o valor 0.0). O erro numérico de ambas as avaliações é somado e o otimizador atualiza os pesos do Policial para que ele não cometa o mesmo erro novamente.

**Fase 2: O Turno do Gerador (Atualizar a rede G)**
* O Gerador não tem acesso direto às imagens reais. Ele aprende exclusivamente tentando "hackear" os pesos do Policial.
* Pegamos as imagens **Falsas** recém-criadas e as passamos novamente pelo Policial, mas desta vez, calculamos o gradiente sob a premissa de que o objetivo do Gerador era arrancar a nota 1.0 (Real) do Policial. Se o Policial der nota baixa, o otimizador pune os pesos do Gerador severamente, forçando-o a aprimorar seus traços anatômicos na iteração seguinte.

*Nota de Engenharia:* Adicionamos um "Árbitro Lógico" nos logs para traduzir as flutuações das derivadas matemáticas em um diagnóstico de domínio (quem está vencendo o jogo de soma zero).

In [ ]:
# ==============================================================================
# BLOCO 5: O LOOP DE TREINAMENTO ADVERSARIAL (CORRIGIDO E OTIMIZADO)
# ==============================================================================
import random

# Definimos 150 épocas para dar tempo ao Gerador de aprender a anatomia
NUM_EPOCHS = 150

print(f"Iniciando a Batalha Adversarial para {NUM_EPOCHS} Épocas...")
print("Estratégias Ativas: Label Smoothing (D) e Label Flipping (5% de Caos)")

G_losses = []
D_losses = []

for epoch in range(NUM_EPOCHS):
    for i, data in enumerate(dataloader, 0):

        # ######################################################################
        # FASE 1: O TURNO DO DISCRIMINADOR (POLICIAL)
        # ######################################################################
        netD.zero_grad()

        # --- Avaliação das Imagens Reais ---
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)

        # LABEL SMOOTHING: O Policial mira em 0.9 em vez de 1.0 para imagens reais
        label_real_D = torch.full((b_size,), 0.9, dtype=torch.float, device=device)

        # LABEL FLIPPING (Buff do Gerador): 5% de chance de mentir para o Policial
        if random.random() < 0.05:
            label_real_D.fill_(0.0) # Diz que a imagem real é falsa

        output = netD(real_cpu).view(-1)
        errD_real = criterion(output, label_real_D)
        errD_real.backward()
        D_x = output.mean().item()

        # --- Avaliação das Imagens Falsas ---
        noise = torch.randn(b_size, LATENT_DIM, 1, 1, device=device)
        fake = netG(noise)

        # O Policial mira em 0.0 para imagens falsas
        label_fake_D = torch.full((b_size,), 0.0, dtype=torch.float, device=device)

        # LABEL FLIPPING (Buff do Gerador): 5% de chance de mentir novamente
        if random.random() < 0.05:
            label_fake_D.fill_(0.9) # Diz que a imagem falsa é real

        output = netD(fake.detach()).view(-1)
        errD_fake = criterion(output, label_fake_D)
        errD_fake.backward()
        D_G_z1 = output.mean().item()

        # Consolida o erro e atualiza o Policial
        errD = errD_real + errD_fake
        optimizerD.step()

        # ######################################################################
        # FASE 2: O TURNO DO GERADOR (FALSIFICADOR)
        # ######################################################################
        netG.zero_grad()

        # CORREÇÃO CRÍTICA: O Gerador SEMPRE quer enganar com perfeição (mira no 1.0)
        label_G = torch.full((b_size,), 1.0, dtype=torch.float, device=device)

        output = netD(fake).view(-1)
        errG = criterion(output, label_G)
        errG.backward()
        D_G_z2 = output.mean().item()

        # Consolida o erro e atualiza o Gerador
        optimizerG.step()

        # ----------------------------------------------------------------------
        # AUDITORIA DE DOMÍNIO
        # ----------------------------------------------------------------------
        if i % 200 == 0:
            status = "Equilíbrio Tenso"
            if D_x > 0.85 and D_G_z2 < 0.15:
                status = "Policial Dominando (Sinal de Alerta)"
            elif D_G_z2 > 0.70:
                status = "Falsificador Dominando"

            print(f"[{epoch+1}/{NUM_EPOCHS}][{i}/{len(dataloader)}] "
                  f"Loss_D: {errD.item():.3f} | Loss_G: {errG.item():.3f} || "
                  f"D(x): {D_x:.3f} | D(G(z)): {D_G_z2:.3f} -> {status}")

        G_losses.append(errG.item())
        D_losses.append(errD.item())

    # --------------------------------------------------------------------------
    # AO FINAL DE CADA ÉPOCA: Auditoria Visual e Checkpoints
    # --------------------------------------------------------------------------
    with torch.no_grad():
        fake_display = netG(fixed_noise).detach().cpu()

    plt.figure(figsize=(8,8))
    plt.axis("off")
    plt.title(f"A Mente do Falsificador: Época {epoch+1}")
    plt.imshow(np.transpose(vutils.make_grid(fake_display, padding=2, normalize=True).cpu(),(1,2,0)))
    plt.show()

    # Salva a imagem gerada no disco
    vutils.save_image(fake_display, f"evolucao_dcgan/anime_epoca_{epoch+1:03d}.png", normalize=True)

    # Salva os pesos da IA a cada 25 épocas e na última época
    if (epoch + 1) % 25 == 0 or (epoch + 1) == NUM_EPOCHS:
        torch.save(netG.state_dict(), f"modelos_salvos/Gerador_Epoca_{epoch+1}.pth")
        # Também podemos salvar o Discriminador caso queiramos retomar o treino depois
        torch.save(netD.state_dict(), f"modelos_salvos/Discriminador_Epoca_{epoch+1}.pth")

print("\nTreinamento Finalizado!")